# Inference from a tabular model

Run Setup before class. The two mini-demos load the Iris data; then run Load
the model and Load the test samples. The demonstrations follow the slides:
First prediction → Prepare the same flower → Check the answer.

## Setup — before class

Use a Jupyter kernel with PyTorch, NumPy and pandas installed. Apart from the first mini-demo, these files run
offline; model weights and saved preprocessing are already included. Open this
notebook from its own folder: the `iris_model` helper, the model files and the
data sit beside it. See `README.md`.

In [ ]:
import pandas as pd
from IPython.display import display
from iris_model import prepare

names = ["Sepal length (cm)", "Sepal width (cm)",
         "Petal length (cm)", "Petal width (cm)"]

def flower_table(row):
    return pd.Series(row.tolist(), index=names, name="Flower").round(1)

## Mini-demo: CSV from a URL

Load all 150 Iris samples into an input tensor and a target tensor. This cell
needs a network connection; the rest of the notebook runs offline.

In [ ]:
import numpy as np
import torch

url = ("https://raw.githubusercontent.com/scikit-learn/"
       "scikit-learn/1.6.1/sklearn/datasets/data/iris.csv")
data = np.loadtxt(url, delimiter=",",
                  skiprows=1, dtype=np.float32)

input_array = data[:, :4]
target_array = data[:, 4].astype(np.int64)
iris_x = torch.from_numpy(input_array)
iris_y = torch.from_numpy(target_array)

## Mini-demo: inspecting iris samples

Columns: sepal length, sepal width, petal length, petal width.
Targets: 0 = setosa, 1 = versicolor, 2 = virginica.

In [ ]:
tuple(iris_x.shape), tuple(iris_y.shape)
# (150, 4), (150,)

In [ ]:
iris_x.dtype, iris_y.dtype
# torch.float32, torch.int64

In [ ]:
iris_x[0], iris_y[0]
# tensor([5.1000, 3.5000, 1.4000, 0.2000]), tensor(0)

## Load the model

This course-trained network has four inputs, one hidden layer with 8 units and
ReLU, and three output scores (setosa, versicolor, virginica). It was trained
on 125 of the 150 Iris samples; the other 25 were kept out of training.

Our `train_iris_model.py` saved two files in this folder: `weights.pt`
holds learned weights and biases; `metadata.json` records feature order,
training means and standard deviations, and class labels. Our `load_model()`
helper rebuilds the network, restores the weights, and reads that JSON into
the Python dictionary `metadata`.

Start with the [README](README.md) for input units,
normalization steps and output interpretation. The supplied
`prepare(measurements, metadata)` helper implements input preparation.

In [ ]:
from iris_model import load_model

model, metadata = load_model()

print(metadata["features"])
# ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
print(metadata["labels"])
# ['setosa', 'versicolor', 'virginica']

## Load the test samples

The same steps as the first mini-demo, now on a local file holding the 25 samples kept
out of training. `test_y` holds the recorded species; keep it away from the model.

In [ ]:
import numpy as np
import torch

data = np.loadtxt("test.csv", delimiter=",",
                  skiprows=1, dtype=np.float32)

test_x = torch.from_numpy(data[:, :4])
test_y = torch.from_numpy(data[:, 4].astype(np.int64))

tuple(test_x.shape), tuple(test_y.shape)
# (25, 4), (25,)

## First prediction

Can these four measurements predict the species? The model's classes are
0 = setosa, 1 = versicolor, 2 = virginica.

In [ ]:
display(flower_table(test_x[0]))

In [ ]:
# The helper prepares this first input. We unpack its work next.
first_x = prepare(test_x[:1], metadata)
with torch.inference_mode():
    first_scores = model(first_x)
    first_probabilities = first_scores.softmax(dim=1)

first_class = first_probabilities.argmax(dim=1).item()
print("Prediction:", metadata["labels"][first_class])
display(pd.Series(
    first_probabilities[0].tolist(), index=metadata["labels"], name="Class score"
).round(3))

Pause for **The first prediction** in the slides. We have a result;
now inspect what the model received and how it produced that label.

## Prepare the same flower

Before training, the preparation script computed a mean and population
standard deviation (`correction=0`) for each measurement using only the 125
training samples. It saved them in `metadata.json` and standardized each
training measurement with `(value - mean) / std`.

Reuse those saved means and standard deviations. Do not compute new statistics
from this one flower or from the test samples.

In [ ]:
# test_x was loaded earlier: 25 flowers, four measurements each.
flower = test_x[:1]

mean = torch.tensor(metadata["mean"])
std = torch.tensor(metadata["std"])
x = (flower - mean) / std

display(pd.DataFrame({
    "Original": flower[0].tolist(),
    "Saved mean": mean.tolist(),
    "Saved std": std.tolist(),
    "Standardized": x[0].tolist(),
}, index=metadata["features"]).round(3))
print("Shape:", x.shape)
print("Matches the first prepared input:", torch.allclose(x, first_x))

## Run the model

One input row is one flower. Evaluation behavior and inference mode are
separate settings: choose prediction behavior and skip bookkeeping for training.

In [ ]:
model.eval()
with torch.inference_mode():
    scores = model(x)
print("Output shape:", scores.shape)
display(pd.Series(
    scores[0].tolist(), index=metadata["labels"], name="Raw score"
).round(3))

## From scores to a label

Softmax makes three nonnegative scores that sum to one. Argmax selects the
winning column's index; the model's label list gives that index its meaning.
This transformation is different from mean/std standardization.

In [ ]:
probabilities = scores.softmax(dim=1)
class_id = probabilities.argmax(dim=1).item()
prediction = metadata["labels"][class_id]

display(pd.DataFrame({
    "Raw score": scores[0].tolist(),
    "After softmax": probabilities[0].tolist(),
}, index=metadata["labels"]).round(3))
print("Winning index:", class_id, "Prediction:", prediction)

## Check the answer

The recorded species was never passed into the model. Compare it now.

In [ ]:
print("Prediction:", prediction)
print("Recorded species:", metadata["labels"][test_y[0].item()])

## Several flowers

Use the first three test samples in file order. Each row still means one flower.

In [ ]:
batch_x = prepare(test_x[:3], metadata)
with torch.inference_mode():
    batch_scores = model(batch_x)
    batch_probabilities = batch_scores.softmax(dim=1)
batch_classes = batch_probabilities.argmax(dim=1)

print("Input:", batch_x.shape, "Output:", batch_scores.shape)
comparison = pd.DataFrame({
    "Recorded species": [metadata["labels"][i] for i in test_y[:3].tolist()],
    "Prediction": [metadata["labels"][i] for i in batch_classes.tolist()],
    "Winning score": batch_probabilities.max(dim=1).values.tolist(),
})
comparison.index.name = "Test sample"
display(comparison.round(3))

The first prediction disagrees with the recorded species. Its two largest
scores were close, but these scores are not calibrated probabilities. The
model's preparation and evaluation notes are in `README.md`.

## Optional: all test samples

All 25 test samples were excluded from training. This is an instructor check,
not a new lesson on data splitting.

In [ ]:
all_x = prepare(test_x, metadata)
with torch.inference_mode():
    all_scores = model(all_x)
all_classes = all_scores.argmax(dim=1)
print("Correct:", int((all_classes == test_y).sum()), "/", len(test_y))